# EHS Report Generation

Runs the structured Stage 2 prompt on a random sample of accident clips and renders
OSHA 300-series incident reports. Operates independently of the ablation sweep.

**Restart the kernel** if `pipeline/classification.py` was just edited.

In [1]:
import os, sys, time
from pathlib import Path

if "__vsc_ipynb_file__" in dir():
    repo_root = str(Path(__vsc_ipynb_file__).parent.parent)
else:
    def _find_repo_root() -> str:
        for p in [Path.cwd()] + list(Path.cwd().parents):
            if (p / "pipeline").is_dir() and (p / "experiments").is_dir():
                return str(p)
        raise RuntimeError("Could not find repo root.")
    repo_root = _find_repo_root()

os.chdir(repo_root)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print("cwd:", os.getcwd())

from dotenv import load_dotenv
load_dotenv(os.path.join(repo_root, ".env"))

from pipeline import classification, ingestion
from evaluation.metrics import load_ground_truth

OUTPUTS = "outputs"
GT      = "data/dataset_mapping.xlsx"
print("Imports ok.")

cwd: /home/grom/DevFiles/Classes/Capstone
Imports ok.


---
## 1 — OSHA Report Formatter

In [2]:
# 1 — OSHA report formatter (uses CoT reasoning for scene assessment)
from datetime import date

SEVERITY_LABEL = {
    'low':      'First Aid Only',
    'medium':   'Medical Treatment Required',
    'high':     'Hospitalisation',
    'critical': 'Life-Threatening / Fatal',
}

def format_osha_report(video_id: str, result, incident_date: str | None = None) -> str:
    """Render a ClassificationResult as an OSHA 300-series incident report.

    The model's chain-of-thought `reasoning` field (scene observation step)
    is surfaced as a 'Scene Assessment' block — it provides the analyst's
    narrative context and grounds all downstream OSHA fields.
    """
    r = result.ehs_report
    cats = ', '.join(f"{c['category']} ({c['confidence']:.0%})" for c in result.categories)
    severity = SEVERITY_LABEL.get(r.get('severity', ''), r.get('severity', 'Unknown'))
    today = incident_date or date.today().strftime('%Y-%m-%d')

    lines = [
        '=' * 68,
        f'  OSHA INCIDENT REPORT — {video_id}',
        '=' * 68,
        f'Date of incident:    {today}',
        f'Time of incident:    {result.incident_start_time or "Unknown"}'
                             f' — {result.incident_end_time or "Unknown"}',
        f'Incident type(s):    {cats}',
        f'Severity:            {severity}',
        '',
    ]

    # Lead with CoT scene assessment when available — this is the model's
    # free-form observation pass before classifying, and gives the richest
    # narrative context for the structured fields below.
    if result.reasoning:
        lines += [
            'SCENE ASSESSMENT (analyst observation)',
            f'  {result.reasoning}',
            '',
        ]

    lines += [
        'WHAT WAS THE EMPLOYEE DOING JUST BEFORE THE INCIDENT?',
        f'  {r.get("pre_incident_activity", "Not captured")}',
        '',
        'WHAT HAPPENED?',
        f'  {r.get("what_happened", result.description)}',
        '',
        'WHAT WAS THE INJURY OR ILLNESS?',
        f'  {r.get("injury_description", "Not captured")}',
        '',
        'WHAT OBJECT OR SUBSTANCE DIRECTLY HARMED THE EMPLOYEE?',
        f'  {r.get("direct_agent") or "N/A"}',
        '',
        'IMMEDIATE ACTIONS',
        f'  {r.get("immediate_actions", "")}',
        '',
        'ROOT CAUSE',
        f'  {r.get("root_cause", "")}',
        '',
        'CONTRIBUTING FACTORS',
        f'  {r.get("contributing_factors", "")}',
        '',
        'CORRECTIVE MEASURES',
        f'  {r.get("corrective_measures", "")}',
        '=' * 68,
    ]
    return '\n'.join(lines)

print('format_osha_report() ready.')

format_osha_report() ready.


---
## 2 — Sample Clips & Run Stage 2

In [3]:
# 2a — Pick 10 random accident clips from the dataset
import random
from pipeline.ingestion import find_local_videos, video_id_from_path

all_paths = find_local_videos('data/videos', originals_only=True)

gt = load_ground_truth(GT)
accident_vids = set(gt[gt['true_binary'] == 'Accident']['video_id_clean'])

accident_paths = [p for p in all_paths if video_id_from_path(p) in accident_vids]
random.seed(42)
selected = random.sample(accident_paths, min(10, len(accident_paths)))

print(f'{len(accident_paths)} accident clips in dataset, sampling {len(selected)}:')
for p in selected:
    print(f'  {video_id_from_path(p):12s}  {p}')

43 accident clips in dataset, sampling 10:
  VID040        /home/grom/DevFiles/Classes/Capstone/data/videos/VID040_Electrical_Work_Safety_Awareness_Training_Electrical_safety_video_animation_TECH_EHS/original.mp4
  VID007        /home/grom/DevFiles/Classes/Capstone/data/videos/VID007_Work_Place_Safety_Video-Part_3_VB_Factory_VB_Engineering_I_Pvt_Ltd/original.mp4
  VID001        /home/grom/DevFiles/Classes/Capstone/data/videos/VID001_Electric_Forklift_Safety_Accident_VB_Factory_VB_Engineering_I_Pvt_Ltd/original.mp4
  VID017        /home/grom/DevFiles/Classes/Capstone/data/videos/VID017_Height_Work_Safety_Awareness_Training_Animated_Safety_Videos_TECH_EHS/original.mp4
  VID015        /home/grom/DevFiles/Classes/Capstone/data/videos/VID015_Woodlift_Safety_Accident_Video_Part-2_VB_Factory_VB_Engineering_I_Pvt_Ltd/original.mp4
  VID014        /home/grom/DevFiles/Classes/Capstone/data/videos/VID014_Woodlift_Safety_Accident_Video_Part-2_VB_Factory_VB_Engineering_I_Pvt_Ltd/original.mp4
  VID00

In [4]:
# 2b — Run Stage 2 directly on each selected clip (skips Stage 1 gate)
import time
from vertexai.generative_models import Part
from pipeline.classification import classify, CLASSIFICATION_PROMPT_STRUCTURED
from pipeline.client import vertex_model

results = {}  # vid_id -> ClassificationResult
for path in selected:
    vid_id = video_id_from_path(path)
    print(f'Processing {vid_id}...', end=' ', flush=True)
    t0 = time.perf_counter()
    try:
        with open(path, 'rb') as f:
            video_part = Part.from_data(data=f.read(), mime_type='video/mp4')
        result = classify(
            video_part=video_part,
            model=vertex_model,
            prompt=CLASSIFICATION_PROMPT_STRUCTURED,
            temperature=0.7,
            top_p=0.95,
            top_k=40,
        )
        results[vid_id] = result
        print(f'ok ({time.perf_counter()-t0:.1f}s) — primary: {result.category}')
    except Exception as e:
        print(f'FAILED: {e}')

print(f'\nDone. {len(results)}/{len(selected)} reports generated.')

Processing VID040... 

/home/grom/DevFiles/Classes/Capstone/.venv/lib/python3.12/site-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


ok (10.8s) — primary: Arc Flash
Processing VID007... ok (8.6s) — primary: Fall
Processing VID001... ok (8.9s) — primary: Fall
Processing VID017... ok (9.1s) — primary: Fall
Processing VID015... ok (11.5s) — primary: Struck by Object
Processing VID014... ok (9.9s) — primary: Trip
Processing VID008... ok (17.0s) — primary: Struck by Object
Processing VID006... ok (10.2s) — primary: Trip
Processing VID034... ok (7.5s) — primary: Struck by Object
Processing VID005... ok (11.0s) — primary: Slip

Done. 10/10 reports generated.


---
## 3 — Display & Export Reports

In [5]:
# 3a — Display OSHA reports
for vid_id, result in results.items():
    print(format_osha_report(vid_id, result))
    print()

  OSHA INCIDENT REPORT — VID040
Date of incident:    2026-03-20
Time of incident:    00:00:01 — 00:00:01
Incident type(s):    Arc Flash (100%), Fire (90%)
Severity:            Hospitalisation

SCENE ASSESSMENT (analyst observation)
  The video clip shows an electrical worker interacting with an open electrical panel, followed by a sudden and intense electrical arc flash incident. The worker is visibly affected by the explosive energy release.

WHAT WAS THE EMPLOYEE DOING JUST BEFORE THE INCIDENT?
  The employee was working on or near live electrical components within an electrical switchgear cabinet, likely performing maintenance, inspection, or troubleshooting.

WHAT HAPPENED?
  While the employee was accessing the internal components of an electrical cabinet, an unexpected electrical arc flash occurred. This event released a significant amount of energy, intense light, and heat, directly exposing the employee to the arc flash.

WHAT WAS THE INJURY OR ILLNESS?
  Potential severe burns

In [6]:
# 3b — Export to text file
import os
os.makedirs('outputs/ehs_reports', exist_ok=True)
output_path = 'outputs/ehs_reports/sample_osha_reports.txt'
with open(output_path, 'w') as f:
    f.write('EHS INCIDENT REPORT SAMPLES\n')
    f.write(f'Generated: {__import__("datetime").date.today()}\n')
    f.write('Pipeline: Gemini 2.5 Flash, structured CoT prompt, temp=0.7\n')
    f.write('=' * 68 + '\n\n')
    for vid_id, result in results.items():
        f.write(format_osha_report(vid_id, result))
        f.write('\n\n')
print(f'Saved {len(results)} reports → {output_path}')

Saved 10 reports → outputs/ehs_reports/sample_osha_reports.txt
